# Temporal possession and frozen learned audio

Private GPU experiment on the same provisional draft holdout as run 02.
Three equal-capacity arms: tracking, current heuristic whistle features,
and current features plus frozen PANNs Cnn6 audio embeddings/scores.
All arms use the mean predictions of seeds 0, 1 and 2, each trained for 60
fixed epochs. No validation early stopping or parameter search. No acoustic
ground truth is invented from possession labels.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import torch

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working/possession_tcn_audio")
WORK.mkdir(parents=True, exist_ok=True)
assert torch.cuda.is_available(), "This experiment requires a free Kaggle GPU."
print("GPU:", torch.cuda.get_device_name(0), "torch", torch.__version__, flush=True)
def sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()
roots = [p.parent for p in INPUT.rglob("requirements-temporal.txt")
         if (p.parent / "temporal_possession/experiment.py").is_file()]
assert len(roots) == 1, roots
CODE = roots[0]
checkpoints = [p for p in INPUT.rglob("*.pth") if sha(p) == "8a17541944f2623a2a0164e072ac2f915dcedca99968083cc44bfbc7eb682275"]
assert len(checkpoints) == 1, [(str(p), p.stat().st_size) for p in INPUT.rglob("*.pth")]
CHECKPOINT = checkpoints[0]
LABELS = CHECKPOINT.parent
assert sha(CHECKPOINT) == "8a17541944f2623a2a0164e072ac2f915dcedca99968083cc44bfbc7eb682275"
audit = json.loads((LABELS / "annotation_audit.json").read_text())
for name in ("train", "validation"):
    assert sha(LABELS / f"{name}.csv") == audit[f"{name}_labels_sha256"]
features = [p for p in INPUT.rglob("features.npz")
            if sha(p) == "dc8c85c46bb381eaa8b1bb6deeb004f458165237bb00b7e89ba4fc4a756f805a"]
assert len(features) == 1
FEATURES = features[0]
videos = list(INPUT.rglob("BAR-ATH_tactical.mp4"))
assert len(videos) == 1
VIDEO = videos[0]
cpu = [p for p in INPUT.rglob("learned.csv") if p.parent.name == "run"
       and "possession_run02" in p.parts]
assert len(cpu) == 1
CPU_BASELINE = cpu[0]
assert sha(CPU_BASELINE) == "a40c89e51a8e7e3e463e870115947585d799b879dc3cc8673e1ebca535d64fe3"
print("Source:", CODE, "Video:", VIDEO, flush=True)
print("Training / validation counts:", audit["training_counts"], audit["validation_counts"], flush=True)


Use the GPU image's preinstalled CUDA PyTorch. Install only the pinned supporting
stack into /tmp, leaving unrelated global packages unchanged. Run all numerical,
media-alignment, leakage and checkpoint tests before encoding or fitting.


In [ ]:
DEPS = Path("/tmp/barca_tcn_audio_python")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
                "--target", str(DEPS), "-r", str(CODE / "requirements-temporal.txt")], check=True)
env = dict(os.environ, PYTHONPATH=str(DEPS) + os.pathsep + str(CODE),
           PYTHONDONTWRITEBYTECODE="1", CUBLAS_WORKSPACE_CONFIG=":4096:8")
def run(*args):
    subprocess.run([sys.executable, *map(str, args)], cwd=CODE, env=env, check=True)
run("-m", "unittest", "discover", "-s", "tests", "-v")


Recreate the bounded label files from the original user JSON and check exact
hashes. Encode audio once with frozen Cnn6; two-second windows every 0.5 seconds
supply coarse context. The existing tonal feature supplies finer timing.
No match audio is used to fine-tune the encoder or create acoustic labels.


In [ ]:
REBUILT = WORK / "recreated_labels"
run("tools/import_review_draft.py", "--plan", LABELS / "plan.json",
    "--review", LABELS / "raw_review.json", "--train-window", "train_h2_05",
    "--validation-window", "test_h1_01", "--include-public-training",
    "--allow-drafts", "--out", REBUILT)
for name in ("train", "validation"):
    assert sha(REBUILT / f"{name}.csv") == sha(LABELS / f"{name}.csv")
AUDIO = WORK / "audio_cache.npz"
run("-m", "temporal_possession", "audio", "--video", VIDEO, "--checkpoint", CHECKPOINT,
    "--features", FEATURES, "--out", AUDIO, "--device", "cuda", "--batch-size", "32")


The complete experiment plan is saved before fitting. PCA/scaling use training
frames only. Each ablation zeroes both values and missingness for excluded
features, uses identical architecture/initialisation/minibatch schedules, and
averages all three fixed seeds. Score validation once per arm after all arms
finish; no checkpoint or decoder is chosen using these scores.


In [ ]:
RUN = WORK / "comparison"
run("-m", "temporal_possession", "compare", "--features", FEATURES, "--audio", AUDIO,
    "--train-labels", REBUILT / "train.csv", "--validation-labels", REBUILT / "validation.csv",
    "--audit", REBUILT / "annotation_audit.json", "--out", RUN, "--device", "cuda",
    "--epochs", "60", "--seeds", "0", "1", "2", "--bootstrap", "1000",
    "--compare", CPU_BASELINE, CODE / "evaluation/results/possession_v6.csv",
    CODE / "evaluation/results/baseline_smooth50.csv")


Reload every arm's saved preprocessing and three checkpoints to regenerate the
full prediction file, without reading labels. Preserve the cache and reports as
private outputs. Scores on this short, draft-labelled window remain provisional;
there is no verified acoustic-event F1 or final-test score.


In [ ]:
for arm in ("tracking", "heuristic_audio", "learned_audio"):
    original = RUN / arm / "predictions.csv"
    regenerated = RUN / arm / "regenerated.csv"
    run("-m", "temporal_possession", "predict", "--features", FEATURES, "--audio", AUDIO,
        "--model", RUN / arm, "--out", regenerated, "--device", "cuda")
    assert original.read_bytes() == regenerated.read_bytes(), arm
    print("Byte-identical regeneration:", arm, sha(original), flush=True)
report = json.loads((RUN / "report.json").read_text())
for name, score in report["scores"].items():
    print(name, "spellF1", round(score["spell_f1_mean"], 4),
          "macroF1", round(score["macro_f1"], 4),
          "D_F1", round(score["per_class"]["D"]["f1"], 4), flush=True)
protocol = dict(evaluation_kind="provisional_draft_annotation_holdout",
                feature_cache_sha256=sha(FEATURES), audio_cache_sha256=sha(AUDIO),
                checkpoint_sha256=sha(CHECKPOINT), annotation_audit=audit,
                regenerated_all_arms=True, acoustic_reference_available=False,
                parameter_search=False, final_test_evaluated=False,
                source_sha256={str(p.relative_to(CODE)): sha(p)
                               for p in (CODE / "temporal_possession").glob("*.py")})
(WORK / "protocol.json").write_text(json.dumps(protocol, indent=2) + chr(10))
print("GPU AUDIO COMPARISON COMPLETE", flush=True)
